In [ ]:
import tempfile
import os
import logging
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import xarray as xr
from IPython.display import display, Markdown, HTML

import oq_wrapper as oqw
import nn_gmm as nng
import ml_tools as mlt

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"

logging.basicConfig(
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    level=logging.WARNING
)

PLOT_IMS = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]


In [ ]:
result_dir = "/Users/claudy/dev/work/data/nn_gmm/results/0815_1857_cv_loc_10Epochs_100Sites"

In [ ]:
# Injected Parameters
result_dir = "/home/claudy/dev/work/data/nn_gmm/results/0811_2115_cv_loc_25Epochs"


In [ ]:
result_dir = Path(result_dir)
empdb_ffp = Path(os.environ["wdata"]) / "nn_gmm/20250616_CS200m_empdb.db"

In [ ]:
display(Markdown("## Inputs\n\n"))
display(Markdown(f"**Result directory**: `{result_dir}`\n\n**Empirical database**: `{empdb_ffp}`"))

In [ ]:
run_config = nng.RunConfig.from_yaml(result_dir / "run_config.yaml")
val_pred_df = pd.read_parquet(result_dir / "val_results.parquet").astype(
    {cur_key: "float32" for cur_key in list(run_config.pred_mean_keys) + list(run_config.pred_std_keys)}).sort_index()
metrics_da = xr.open_dataarray(result_dir / "metrics.nc")

In [ ]:
val_record_int_ids = val_pred_df.index.values.astype(int)

with nng.IMDB(run_config.imdb_ffp, readonly=True) as imdb:
    event_df = imdb.get_event_df()
    site_df = imdb.get_site_df()
    val_sim_df = imdb.get_im_data_tmp_table(run_config.ims, val_record_int_ids).sort_index()
    val_record_info_df = imdb.get_record_info_df(record_int_ids=val_record_int_ids)
    val_site_event_df = imdb.get_site_event_df()

val_sim_df["event_int_id"] = val_record_info_df.loc[val_sim_df.index, "event_int_id"].values
val_sim_df["site_int_id"] = val_record_info_df.loc[val_sim_df.index, "site_int_id"].values
val_sim_df["site_event_int_id"] = nng.utils.get_site_event_int_id(val_sim_df.site_int_id, val_sim_df.event_int_id)
val_sim_df["magnitude"] = event_df.loc[val_sim_df["event_int_id"], "magnitude"].values
val_sim_df["rrup"] = val_site_event_df.loc[val_sim_df["site_event_int_id"], "rrup"].values

val_pred_df["site_int_id"] = val_record_info_df.loc[val_pred_df.index, "site_int_id"].values

assert val_pred_df.index.equals(val_sim_df.index)

In [ ]:
with nng.EmpiricalDB(empdb_ffp, readonly=True) as empdb:
    val_emp_df = empdb.get_gm_params_tmp_table(val_pred_df.index.values.astype(int))

assert val_emp_df.index.equals(val_pred_df.index)
assert val_emp_df.index.equals(val_sim_df.index)

In [ ]:
res_df = pd.DataFrame(data=np.log(val_sim_df[run_config.ims].values) - val_pred_df[run_config.pred_mean_keys].values, index=val_pred_df.index, columns=run_config.ims)
res_df["cv_iter"] = val_pred_df["cv_iter"]
emp_res_df = pd.DataFrame(data=np.log(val_sim_df[nng.constants.PSA_KEYS].values) - val_emp_df[nng.constants.GMM_PSA_MEAN_KEYS].values, index=val_emp_df.index, columns=nng.constants.PSA_KEYS)

In [ ]:
cv_res_dfs = {cv_key: res_df.loc[res_df.cv_iter == cv_key] for cv_key in res_df.cv_iter.unique()}

## Metrics

In [ ]:
metric_keys = ["loss_hist", "mse_hist", "mean_sigma_hist"]

In [ ]:
for cur_metric in metric_keys:
    cur_train_df = metrics_da.loc[:, :, cur_metric + "_train"].to_pandas()
    cur_val_df = metrics_da.loc[:, :, cur_metric + "_val"].to_pandas()

    display(Markdown(f"### {cur_metric.replace('_', ' ').replace('hist', 'History').title()}"))
    display(Markdown("#### Training Data"))
    display(HTML(cur_train_df.T.describe().to_html()))

    display(Markdown("#### Validation Data"))
    display(HTML(cur_val_df.T.describe().to_html()))

In [ ]:
metric_limits = {
    "loss_hist": (-1.0, -0.5),
    "mse_hist": (0, 0.2),
    "mean_sigma_hist": (0, 0.5)
}

### CV-Metric Comparison

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(metric_keys) * 2, 2, -1, (8, 6))

for ix, cur_metric_key in enumerate(metric_keys):
	ax1, ax2 = axs[ix * 2], axs[ix * 2 + 1]
    
    # Get current metric values
	cur_train_metric_df = metrics_da.loc[:, :, cur_metric_key + "_train"].to_pandas().T
	cur_val_metric_df = metrics_da.loc[:, :, cur_metric_key + "_val"].to_pandas().T
    
    # Compute mean and std
	cur_mean = cur_train_metric_df.mean(axis=1)
	cur_std = cur_train_metric_df.std(axis=1)
	
	cur_val_mean = cur_val_metric_df.mean(axis=1)
	cur_val_std = cur_val_metric_df.std(axis=1)

	y_min, y_max = metric_limits[cur_metric_key]	

    # Plot
	ax1.plot(cur_train_metric_df.index.values, cur_train_metric_df, c="gray", linewidth=1.0, linestyle="--")
	ax1.plot(cur_mean.index.values, cur_mean, c="black", linewidth=2.0)
	ax1.set_ylim(y_min, y_max)
	ax1.set_title(f"Training {cur_metric_key}")
	ax1.set_xlim(cur_train_metric_df.index.values.min(), cur_train_metric_df.index.values.max())
	ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    
	ax2.plot(cur_val_metric_df.index.values, cur_val_metric_df, c="gray", linewidth=1.0, linestyle="--")
	ax2.plot(cur_val_mean.index.values, cur_val_mean, c="black", linewidth=2.0)
	ax2.set_ylim(y_min, y_max)
	ax2.set_title(f"Validation {cur_metric_key}")
	ax2.set_xlim(cur_train_metric_df.index.values.min(), cur_train_metric_df.index.values.max())
	ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    
fig.tight_layout()

### Comparison of median metrics

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(metric_keys), 1, -1, (10, 6))

for cur_ax, cur_metric_key in zip(axs, metric_keys):
    # Get current metric values
	cur_train_metric_df = metrics_da.loc[:, :, cur_metric_key + "_train"].to_pandas().T
	cur_val_metric_df = metrics_da.loc[:, :, cur_metric_key + "_val"].to_pandas().T
    
    # Get the median and 16th/84th percentiles
	train_median = cur_train_metric_df.median(axis=1)
	train_16, train_84 = np.percentile(cur_train_metric_df, [16, 84], axis=1)
	
	val_median = cur_val_metric_df.median(axis=1)
	val_16, val_84 = np.percentile(cur_val_metric_df, [16, 84], axis=1)
    
	# Get y-axis limits
	y_min, y_max = metric_limits[cur_metric_key]

	# Plot
	cur_ax.plot(cur_train_metric_df.index.values, train_median, label="Training - Median", c="tab:blue")
	cur_ax.fill_between(cur_train_metric_df.index.values, train_16, train_84, alpha=0.5, color="tab:blue")
	cur_ax.plot(cur_train_metric_df.index.values, train_16, label="Training - 16th & 84th", linestyle="--", c="tab:blue", linewidth=1.0)
	cur_ax.plot(cur_train_metric_df.index.values, train_84, linestyle="--", c="tab:blue", linewidth=1.0)

	
	cur_ax.plot(cur_val_metric_df.index.values, val_median, label="Validation - Median", c="tab:orange")
	cur_ax.fill_between(cur_val_metric_df.index.values, val_16, val_84, alpha=0.5, color="tab:orange")
	cur_ax.plot(cur_val_metric_df.index.values, val_16, label="Validation - 16th & 84th", linestyle="--", c="tab:orange", linewidth=1.0)
	cur_ax.plot(cur_val_metric_df.index.values, val_84, linestyle="--", c="tab:orange", linewidth=1.0)
	
	cur_ax.set_ylim(y_min, y_max)
	cur_ax.set_xlim(cur_train_metric_df.index.values.min(), cur_train_metric_df.index.values.max())
	cur_ax.set_xlabel("Epoch")
	cur_ax.set_ylabel(f"{cur_metric_key}")
	cur_ax.set_title(f"{cur_metric_key}")
	cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	cur_ax.legend()
	
	fig.tight_layout()

## Model Bias & Residual Standard Deviation

### Total

In [ ]:
bias_ylim = (-0.8, 0.8)
std_ylim = (0.0, 0.8)

In [ ]:
model_bias, res_std = res_df[run_config.ims].mean(axis=0), res_df[run_config.ims].std(axis=0)
emp_model_bias, emp_res_std = emp_res_df[run_config.ims].mean(axis=0), emp_res_df[run_config.ims].std(axis=0)

In [ ]:
fig, ax1, ax2, ax3, ax4 = nng.plot_utils.get_bias_residual_fig(figsize=(16, 6), bias_y_axis_limits=bias_ylim, std_y_axis_limits=std_ylim)

ax1.plot(nng.constants.PSA_PERIODS, model_bias[nng.constants.PSA_KEYS].values, c="b", label="NN")
ax1.plot(nng.constants.PSA_PERIODS, emp_model_bias[nng.constants.PSA_KEYS].values, c="g", label="Empirical")

ax1.legend()

if run_config.im_set == "all":
    ax2.scatter(nng.constants.NON_PSA_IMS, model_bias[nng.constants.NON_PSA_IMS].values, c="b")

ax3.plot(nng.constants.PSA_PERIODS, res_std[nng.constants.PSA_KEYS].values, c="b")
ax3.plot(nng.constants.PSA_PERIODS, emp_res_std[nng.constants.PSA_KEYS].values, c="g")


if run_config.im_set == "all":
    ax4.scatter(nng.constants.NON_PSA_IMS, res_std[nng.constants.NON_PSA_IMS].values, c="b")

### CV

In [ ]:
fig, ax1, ax2, ax3, ax4 =  nng.plot_utils.get_bias_residual_fig(figsize=(16, 6), bias_y_axis_limits=bias_ylim, std_y_axis_limits=std_ylim)

cv_res_mean_values = pd.concat([cur_cv_val_res[run_config.ims].mean().to_frame(cv_key).T for cv_key, cur_cv_val_res in cv_res_dfs.items()], axis=0)
cv_bias_std = cv_res_mean_values.std(axis=0)

# Bias
ax1.semilogx(np.asarray(nng.constants.PSA_PERIODS)[:, None], cv_res_mean_values[nng.constants.PSA_KEYS].T, c="gray", alpha=0.75, linestyle="--", linewidth=1.0)
ax1.semilogx(nng.constants.PSA_PERIODS, model_bias[nng.constants.PSA_KEYS].values, c="b", label="Mean")
ax1.fill_between(nng.constants.PSA_PERIODS, model_bias.loc[nng.constants.PSA_KEYS] - cv_bias_std.loc[nng.constants.PSA_KEYS], 
                 model_bias.loc[nng.constants.PSA_KEYS] + cv_bias_std.loc[nng.constants.PSA_KEYS], alpha=0.2, color="blue")

# Bias - Non-pSA
if run_config.im_set == "all":
    ax2.errorbar(nng.constants.NON_PSA_IMS, model_bias.loc[nng.constants.NON_PSA_IMS], yerr=cv_bias_std.loc[nng.constants.NON_PSA_IMS], fmt="o", ms=5)
    ax2.xaxis.set_tick_params(rotation=90)

    # Individual
    for _, cur_row in cv_res_mean_values.iterrows():
        ax2.scatter(nng.constants.NON_PSA_IMS, cur_row.loc[nng.constants.NON_PSA_IMS], c="gray",
                    alpha=0.5, marker="x", s=2.5)


cv_val_res_std_values = pd.concat([cur_cv_val_res[run_config.ims].std().to_frame(cv_key).T for cv_key, cur_cv_val_res in cv_res_dfs.items()], axis=0)
cv_val_res_std_std = cv_val_res_std_values.std(axis=0)

# Individual CV residual standard deviations
ax3.semilogx(np.asarray(nng.constants.PSA_PERIODS)[:, None], cv_val_res_std_values[nng.constants.PSA_KEYS].T, c="gray", alpha=0.75, linestyle="--",linewidth=1.0)

# Residual Standard Deviation - pSA
ax3.plot(nng.constants.PSA_PERIODS, res_std[nng.constants.PSA_KEYS].values, c="b")
ax3.fill_between(nng.constants.PSA_PERIODS, res_std.loc[nng.constants.PSA_KEYS] - cv_val_res_std_std.loc[nng.constants.PSA_KEYS], res_std.loc[nng.constants.PSA_KEYS] + cv_val_res_std_std.loc[nng.constants.PSA_KEYS], alpha=0.2, color="blue")
ax3.set_ylim(std_ylim)

# Residual Standard Deviation - Non-pSA
if run_config.im_set == "all":
    ax4.errorbar(nng.constants.NON_PSA_IMS, res_std.loc[nng.constants.NON_PSA_IMS], yerr=cv_val_res_std_std.loc[nng.constants.NON_PSA_IMS], fmt="o", ms=5)
    ax4.xaxis.set_tick_params(rotation=90)
    ax4.set_ylim(ax3.get_ylim())

    # Individual
    for _, cur_row in cv_val_res_std_values.iterrows():
        ax4.scatter(nng.constants.NON_PSA_IMS, cur_row.loc[nng.constants.NON_PSA_IMS], c="gray",
                    alpha=0.5, marker="x", s=2.5)

### Magnitude

In [ ]:
mag_bins = [5.5, 6.5, 7.5, 8.5]
mag_bin_labels = [f"$M_w$: {mag_bins[i]} - {mag_bins[i+1]}" for i in range(len(mag_bins) - 1)]
mag_bin_colors = ["blue", "purple", "red"]

In [ ]:
val_sim_df["mag_bin"] = pd.cut(val_sim_df["magnitude"], bins=mag_bins, labels=mag_bin_labels, include_lowest=True)

In [ ]:
fig, ax1, ax2, ax3, ax4 = nng.plot_utils.get_bias_residual_fig(figsize=(16, 6), bias_y_axis_limits=bias_ylim, std_y_axis_limits=std_ylim)

ax1.plot(nng.constants.PSA_PERIODS, model_bias[nng.constants.PSA_KEYS].values, c="k")
ax1.plot(nng.constants.PSA_PERIODS, emp_model_bias[nng.constants.PSA_KEYS].values, c="k", linestyle="--")

for i, cur_mag_bin in enumerate(mag_bin_labels):
    cur_record_ids = val_sim_df.loc[val_sim_df["mag_bin"] == cur_mag_bin].index.values
    cur_model_bias = res_df.loc[cur_record_ids, nng.constants.PSA_KEYS].mean(axis=0)
    cur_emp_model_bias = emp_res_df.loc[cur_record_ids].mean(axis=0)
    ax1.plot(nng.constants.PSA_PERIODS, cur_emp_model_bias[nng.constants.PSA_KEYS].values, c=mag_bin_colors[i], linestyle="--")
    ax1.plot(nng.constants.PSA_PERIODS, cur_model_bias[nng.constants.PSA_KEYS].values, c=mag_bin_colors[i], label=f"{cur_mag_bin}, N={len(cur_record_ids)}")

ax1.legend()

if run_config.im_set == "all":
    ax2.scatter(nng.constants.NON_PSA_IMS, model_bias[nng.constants.NON_PSA_IMS].values, c="b")

ax3.plot(nng.constants.PSA_PERIODS, res_std[nng.constants.PSA_KEYS].values, c="k")
ax3.plot(nng.constants.PSA_PERIODS, emp_res_std[nng.constants.PSA_KEYS].values, c="k", linestyle="--")

for i, cur_mag_bin in enumerate(mag_bin_labels):
    cur_record_ids = val_sim_df.loc[val_sim_df["mag_bin"] == cur_mag_bin].index.values
    cur_res_std = res_df.loc[cur_record_ids, nng.constants.PSA_KEYS].std(axis=0)
    cur_emp_res_std = emp_res_df.loc[cur_record_ids].std(axis=0)
    ax3.plot(nng.constants.PSA_PERIODS, cur_emp_res_std[nng.constants.PSA_KEYS].values, c=mag_bin_colors[i], linestyle="--")
    ax3.plot(nng.constants.PSA_PERIODS, cur_res_std[nng.constants.PSA_KEYS].values, c=mag_bin_colors[i], label=cur_mag_bin)


if run_config.im_set == "all":
    ax4.scatter(nng.constants.NON_PSA_IMS, res_std[nng.constants.NON_PSA_IMS].values, c="b")

### $R_{Rup}$

In [ ]:
rrup_bins = [0, 30, 100, 500]
rrup_bin_labels = ["$R_{Rup}$ 0 - 30", "$R_{Rup}$ 30 - 100", "$R_{Rup}$ 100 - 500"]
rrup_bin_colors = ["blue", "purple", "red"]

In [ ]:
val_sim_df["rrup_bin"] = pd.cut(val_sim_df["rrup"], bins=rrup_bins, labels=rrup_bin_labels, include_lowest=True)

In [ ]:
fig, ax1, ax2, ax3, ax4 = nng.plot_utils.get_bias_residual_fig(figsize=(16, 6), bias_y_axis_limits=(-0.8, 0.8), std_y_axis_limits=(0.0, 0.75))

ax1.plot(nng.constants.PSA_PERIODS, model_bias[nng.constants.PSA_KEYS].values, c="k")
ax1.plot(nng.constants.PSA_PERIODS, emp_model_bias[nng.constants.PSA_KEYS].values, c="k", linestyle="--")

for i, cur_rrup_bin in enumerate(rrup_bin_labels):
    cur_record_ids = val_sim_df.loc[val_sim_df["rrup_bin"] == cur_rrup_bin].index.values
    cur_model_bias = res_df.loc[cur_record_ids, nng.constants.PSA_KEYS].mean(axis=0)
    cur_emp_model_bias = emp_res_df.loc[cur_record_ids].mean(axis=0)
    ax1.plot(nng.constants.PSA_PERIODS, cur_emp_model_bias[nng.constants.PSA_KEYS].values, c=rrup_bin_colors[i], linestyle="--")
    ax1.plot(nng.constants.PSA_PERIODS, cur_model_bias[nng.constants.PSA_KEYS].values, c=rrup_bin_colors[i], label=f"{cur_rrup_bin} (km), N={len(cur_record_ids)}")

ax1.legend()

if run_config.im_set == "all":
    ax2.scatter(nng.constants.NON_PSA_IMS, model_bias[nng.constants.NON_PSA_IMS].values, c="b")

ax3.plot(nng.constants.PSA_PERIODS, res_std[nng.constants.PSA_KEYS].values, c="k")
ax3.plot(nng.constants.PSA_PERIODS, emp_res_std[nng.constants.PSA_KEYS].values, c="k", linestyle="--")

for i, cur_rrup_bin in enumerate(rrup_bin_labels):
    cur_record_ids = val_sim_df.loc[val_sim_df["rrup_bin"] == cur_rrup_bin].index.values
    cur_res_std = res_df.loc[cur_record_ids, nng.constants.PSA_KEYS].std(axis=0)
    cur_emp_res_std = emp_res_df.loc[cur_record_ids].std(axis=0)
    ax3.plot(nng.constants.PSA_PERIODS, cur_emp_res_std[nng.constants.PSA_KEYS].values, c=rrup_bin_colors[i], linestyle="--")
    ax3.plot(nng.constants.PSA_PERIODS, cur_res_std[nng.constants.PSA_KEYS].values, c=rrup_bin_colors[i], label=cur_rrup_bin)


if run_config.im_set == "all":
    ax4.scatter(nng.constants.NON_PSA_IMS, res_std[nng.constants.NON_PSA_IMS].values, c="b")

## Model Trends

### Magnitude

In [ ]:
# Define fixed inputs and limits for validation data points
fixed_inputs = dict(vs30=500,
    z1p0=0.13,
    z2p5=1,
    tect_type=nng.constants.REVERSE_TECTONIC_TYPE_MAPPING[oqw.constants.TectType.ACTIVE_SHALLOW],
    rake=10,
    dip=60,
    dtop=0,
    dbottom=15,
    hypo_depth=8.5,
    # Chch
    nztm_x=1570373.11078697,
    nztm_y=5179891.74707139
)
mag_limits = {
        "vs30": (25, 25),
        "z1p0": (0.25, 0.25),
        "rrup": (5, 5),
}

#### Short Distance ($R_{Rup} = 25$ km)

In [ ]:
short_dist_fixed_inputs = {"rrup": 25, "rjb": 25, "rx": 25} | fixed_inputs

nng.plots.magnitude_trend_plot(result_dir, val_sim_df, short_dist_fixed_inputs, mag_limits, val_record_int_ids, device, cv=True, major_line_width=2.0, minor_line_width=1.5);

#### Moderate Distance ($R_{Rup} = 75$ km)

In [ ]:
moderate_dist_fixed_inputs = {"rrup": 75, "rjb": 75, "rx": 75} | fixed_inputs

nng.plots.magnitude_trend_plot(result_dir, val_sim_df, moderate_dist_fixed_inputs, mag_limits, val_record_int_ids, device, cv=True, major_line_width=2.0, minor_line_width=1.5);

#### Long Distance ($R_{Rup} = 125$ km)

In [ ]:
long_dist_fixed_inputs = {"rrup": 125, "rjb": 125, "rx": 125} | fixed_inputs

nng.plots.magnitude_trend_plot(result_dir, val_sim_df, long_dist_fixed_inputs, mag_limits, val_record_int_ids, device, cv=True, major_line_width=2.0, minor_line_width=1.5);

### Source-To-Site Distance

In [ ]:
rrup_limits = {
        "vs30": (25, 25),
        "z1p0": (0.25, 0.25),
        "magnitude": (0.1, 0.1),
}

#### Magnitude 6

In [ ]:
moderate_mag_fixed_inputs = {"magnitude": 6.0} | fixed_inputs

nng.plots.rrup_trend_plot(result_dir, val_sim_df, moderate_mag_fixed_inputs, rrup_limits, val_record_int_ids, device, cv=True, major_line_width=2.0, minor_line_width=1.5);

#### Magnitude 7

In [ ]:
moderate_mag_fixed_inputs = {"magnitude": 7.0} | fixed_inputs

nng.plots.rrup_trend_plot(result_dir, val_sim_df, moderate_mag_fixed_inputs, rrup_limits, val_record_int_ids, device, cv=True, major_line_width=2.0, minor_line_width=1.5);

#### Magnitude 8

In [ ]:
moderate_mag_fixed_inputs = {"magnitude": 8.0} | fixed_inputs

nng.plots.rrup_trend_plot(result_dir, val_sim_df, moderate_mag_fixed_inputs, rrup_limits, val_record_int_ids, device, cv=True, major_line_width=2.0, minor_line_width=1.5);